# pass@k [Step 08.03 - What "solved" means when you can retry]

> **MLCourse - Agentic AI - Agent Patterns**

A single pass/fail per task hides the question that actually matters in production:
**if I let the agent try k times, how often does at least one attempt succeed?**

That is **pass@k**. It comes from code generation (Codex/HumanEval) and it is now
standard for agents, because agents are stochastic and retries are cheap.

```
pass@1  = P(a single attempt succeeds)
pass@k  = P(at least one of k attempts succeeds)
```

pass@k rises with k by construction. It is not a better score - it is a **different
question**, and it is only meaningful if you actually *can* retry: you must have a
way to tell a good attempt from a bad one at runtime (a test suite, a schema check,
a verifier). If you cannot detect failure in production, your operating point is
pass@1 and quoting pass@5 is misleading.

> **Do not confuse this with tau-bench's `pass^k`** (notebook 01), which requires
> **all** k attempts to succeed. pass@k is optimistic; pass^k is pessimistic.

### Key takeaways

- Estimate pass@k from **n >= k samples** with the unbiased estimator, not by running
  exactly k attempts and looking.
- Report **k, n and the task count** next to every pass@k number.
- A large pass@1 -> pass@k gap means the agent *can* do the task but is unreliable.
  That is a different (and often easier) problem than one it cannot do at all.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 3.5          # 4 tasks x 3 samples x up to 3 steps: pace generously
print("PACE =", PACE)

PACE = 3.5


### The benchmark: world, tools, tasks, grader


In [ ]:
# Four pieces, and they must be separable. If the grader lives inside the agent,
# you are not running a benchmark, you are running a demo.

import re

# 1. THE WORLD -----------------------------------------------------------------
EMPLOYEES = {
    "priya":  dict(name="Priya",  dept="Engineering", salary=82000),
    "marco":  dict(name="Marco",  dept="Engineering", salary=61000),
    "ana":    dict(name="Ana",    dept="Design",      salary=74000),
    "kofi":   dict(name="Kofi",   dept="Engineering", salary=69000),
    "lena":   dict(name="Lena",   dept="Design",      salary=71000),
}

TOOL_CALLS = {"n": 0}


# 2. THE TOOLS -----------------------------------------------------------------
def tool_lookup(arg):
    """lookup(name) -> the employee record, or an error string."""
    TOOL_CALLS["n"] += 1
    rec = EMPLOYEES.get(arg.strip().lower().strip('"\''))
    if not rec:
        return "ERROR: no employee named %r" % arg
    return "name=%s dept=%s salary=%d" % (rec["name"], rec["dept"], rec["salary"])


def tool_count_dept(arg):
    """count_dept(department) -> how many people work in it."""
    TOOL_CALLS["n"] += 1
    d = arg.strip().lower().strip('"\'')
    n = sum(1 for r in EMPLOYEES.values() if r["dept"].lower() == d)
    return "count=%d" % n


TOOLS = {"lookup": tool_lookup, "count_dept": tool_count_dept}

TOOL_DOC = """You have exactly two tools:
  lookup(name)            -> name, department and salary of one employee
  count_dept(department)  -> how many people work in that department

To use a tool, reply with ONLY one line:
CALL: <tool_name>(<argument>)

When you know the answer, reply with ONLY one line:
FINAL: <answer>

Never output both. Never explain. One line per turn."""


# 3. THE TASKS -----------------------------------------------------------------
# Each task carries its OWN grader. Different tasks need different checks, and
# pretending otherwise is how benchmarks end up measuring string formatting.

def num_grader(expected, tol=0.5):
    def check(answer):
        nums = re.findall(r"-?\d+(?:\.\d+)?", (answer or "").replace(",", ""))
        if not nums:
            return False
        return abs(float(nums[-1]) - expected) <= tol
    return check


def word_grader(expected):
    def check(answer):
        return expected.lower() in (answer or "").lower()
    return check


TASKS = [
    dict(id="salary",   q="What is Priya's salary?",
         grade=num_grader(82000), min_tools=1),
    dict(id="headcount", q="How many people work in Engineering?",
         grade=num_grader(3), min_tools=1),
    dict(id="combined", q="What is the combined salary of Priya and Marco? Give a single number.",
         grade=num_grader(143000), min_tools=2),
    dict(id="compare",  q="Who earns more, Ana or Marco? Answer with only the name.",
         grade=word_grader("ana"), min_tools=2),
]

print("%d tasks, %d tools, %d employees" % (len(TASKS), len(TOOLS), len(EMPLOYEES)))


### The agent under test


In [ ]:
# A minimal ReAct-shaped loop: the model either calls a tool or answers. Capped at
# MAX_STEPS, because an unbounded agent in a benchmark is an unbounded bill.

MAX_STEPS = 3
CALL_RE = re.compile(r"CALL:\s*(\w+)\s*\((.*?)\)", re.S)
FINAL_RE = re.compile(r"FINAL:\s*(.+)", re.S)


def run_agent(question, temperature=0.0, verbose=False):
    """Run one attempt. Returns a dict - never raises, so one bad task cannot
    abort a benchmark run halfway through."""
    llm = make_llm(temperature=temperature, max_tokens=120)
    TOOL_CALLS["n"] = 0
    transcript = []
    messages = [("system", "You are a precise data assistant.\n\n" + TOOL_DOC),
                ("user", question)]
    tin = tout = 0
    answer = None
    steps = 0

    for steps in range(1, MAX_STEPS + 1):
        msg = safe_invoke(llm, messages)
        u = msg.usage_metadata or {}
        tin += u.get("input_tokens", 0)
        tout += u.get("output_tokens", 0)
        text = msg.content.strip()
        transcript.append(text)
        if verbose:
            print("  [step %d] %s" % (steps, text.replace("\n", " ")[:100]))

        fin = FINAL_RE.search(text)
        call = CALL_RE.search(text)
        # A model that emits both is ambiguous; prefer the tool call, since a
        # premature FINAL is the more common failure.
        if call and (not fin or call.start() < fin.start()):
            tool, arg = call.group(1), call.group(2)
            result = TOOLS[tool](arg) if tool in TOOLS else "ERROR: no such tool %r" % tool
            if verbose:
                print("           -> %s" % result)
            messages = messages + [("assistant", text), ("user", "TOOL RESULT: " + result)]
            continue
        if fin:
            answer = fin.group(1).strip().splitlines()[0].strip()
            break
        # Neither: nudge once, then give up.
        messages = messages + [("assistant", text),
                               ("user", "Reply with ONLY one line: CALL: ... or FINAL: ...")]

    return dict(answer=answer, steps=steps, tool_calls=TOOL_CALLS["n"],
                tokens=tin + tout, tin=tin, tout=tout, transcript=transcript)


### 1. The estimator

The naive method - "run k attempts, did any pass?" - is a *biased, high-variance*
estimate of pass@k. The standard fix (Chen et al., 2021) is to draw **n samples per
task with c successes**, then compute the exact probability that a random subset of
size k contains at least one success:

```
pass@k  =  1  -  C(n - c, k) / C(n, k)
```

If fewer than k failures exist (`n - c < k`), the value is exactly 1. Note this needs
no simulation and no extra calls - it reuses the same n samples for every k <= n.

In [5]:
from math import comb


def pass_at_k(n, c, k):
    """Unbiased pass@k for a task with c successes out of n samples."""
    if k > n:
        raise ValueError("k must be <= n")
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)


print("sanity checks")
print("  n=3 c=0 -> pass@1 %.3f  pass@3 %.3f" % (pass_at_k(3, 0, 1), pass_at_k(3, 0, 3)))
print("  n=3 c=1 -> pass@1 %.3f  pass@3 %.3f" % (pass_at_k(3, 1, 1), pass_at_k(3, 1, 3)))
print("  n=3 c=2 -> pass@1 %.3f  pass@3 %.3f" % (pass_at_k(3, 2, 1), pass_at_k(3, 2, 3)))
print("  n=3 c=3 -> pass@1 %.3f  pass@3 %.3f" % (pass_at_k(3, 3, 1), pass_at_k(3, 3, 3)))
print()
print("Note c=1 of 3: pass@1 is 0.333 but pass@3 is 1.000 - one lucky attempt out")
print("of three guarantees a hit if you take all three. That gap is the value of")
print("retrying, and it is exactly what pass@k is for.")

sanity checks
  n=3 c=0 -> pass@1 0.000  pass@3 0.000
  n=3 c=1 -> pass@1 0.333  pass@3 1.000
  n=3 c=2 -> pass@1 0.667  pass@3 1.000
  n=3 c=3 -> pass@1 1.000  pass@3 1.000

Note c=1 of 3: pass@1 is 0.333 but pass@3 is 1.000 - one lucky attempt out
of three guarantees a hit if you take all three. That gap is the value of
retrying, and it is exactly what pass@k is for.


### 2. Sampling

We need **temperature > 0**, or every sample is the same and pass@k collapses to
pass@1. This is the point most people miss: pass@k measures the *diversity* of the
agent's attempts as much as its competence.

n=3 samples per task, 4 tasks. Small on purpose - the free tier is shared.

In [6]:
N_SAMPLES = 3
TEMP = 0.8

samples = {t["id"]: [] for t in TASKS}
for t in TASKS:
    for i in range(N_SAMPLES):
        r = run_agent(t["q"], temperature=TEMP)
        ok = bool(t["grade"](r["answer"]))
        samples[t["id"]].append(dict(passed=ok, answer=r["answer"],
                                     tools=r["tool_calls"], tokens=r["tokens"]))
        print("%-11s sample %d/%d -> %-5s answer=%-20s tools=%d"
              % (t["id"], i + 1, N_SAMPLES, "PASS" if ok else "FAIL",
                 str(r["answer"])[:20], r["tool_calls"]))
    print()

salary      sample 1/3 -> PASS  answer=82000                tools=1


salary      sample 2/3 -> PASS  answer=82000                tools=1


salary      sample 3/3 -> PASS  answer=82000                tools=1



headcount   sample 1/3 -> PASS  answer=3                    tools=1


headcount   sample 2/3 -> PASS  answer=3                    tools=1


...

  [rate limit] sleeping 2.0s (attempt 1/5)


compare     sample 2/3 -> PASS  answer=Ana                  tools=2


  [rate limit] sleeping 2.0s (attempt 1/5)


  [rate limit] sleeping 2.0s (attempt 1/5)


compare     sample 3/3 -> PASS  answer=Ana                  tools=2



In [7]:
print("%-12s %6s %6s %10s %10s   %s" % ("task", "n", "c", "pass@1", "pass@3", "answers seen"))
print("-" * 88)
p1s, p3s = [], []
for t in TASKS:
    s = samples[t["id"]]
    c = sum(1 for x in s if x["passed"])
    p1 = pass_at_k(N_SAMPLES, c, 1)
    p3 = pass_at_k(N_SAMPLES, c, 3)
    p1s.append(p1)
    p3s.append(p3)
    seen = sorted({str(x["answer"])[:14] for x in s})
    print("%-12s %6d %6d %10.3f %10.3f   %s"
          % (t["id"], N_SAMPLES, c, p1, p3, " | ".join(seen)))
print("-" * 88)
print("%-12s %6s %6s %10.3f %10.3f" % ("MEAN", "", "", sum(p1s) / len(p1s), sum(p3s) / len(p3s)))
print()
print("MEASURED: pass@1 = %.3f, pass@3 = %.3f (n=%d samples/task, %d tasks, T=%.1f)"
      % (sum(p1s) / len(p1s), sum(p3s) / len(p3s), N_SAMPLES, len(TASKS), TEMP))

task              n      c     pass@1     pass@3   answers seen
----------------------------------------------------------------------------------------
salary            3      3      1.000      1.000   82000
headcount         3      3      1.000      1.000   3
combined          3      3      1.000      1.000   143000
compare           3      3      1.000      1.000   Ana
----------------------------------------------------------------------------------------
MEAN                            1.000      1.000

MEASURED: pass@1 = 1.000, pass@3 = 1.000 (n=3 samples/task, 4 tasks, T=0.8)


### 3. Reading the gap

Look at the per-task rows, especially the `answers seen` column.

- **c = n (all pass).** pass@1 = pass@3 = 1. The task is solved reliably. Nothing to
  gain from retries.
- **c = 0.** pass@k = 0 for every k. Retrying cannot help; the agent lacks the
  capability, the tool, or the prompt. This is where to spend engineering effort.
- **0 < c < n.** The interesting case. The agent *can* do it and sometimes does not.
  The pass@1 -> pass@3 gap quantifies how much a retry-with-verification would buy
  you, and the `answers seen` column usually shows exactly what goes wrong.

**And the honest caveat:** with n=3 the only possible per-task pass@1 values are
0, 1/3, 2/3 and 1. That is a very coarse instrument. Published pass@k numbers use
n of 10-100. Ours demonstrates the estimator correctly; it does not measure this
agent precisely, and the difference matters.

In [8]:
print("what retrying would buy, per task")
print("-" * 58)
for t, p1, p3 in zip(TASKS, p1s, p3s):
    gap = p3 - p1
    if p1 == 1.0:
        verdict = "already reliable - retries buy nothing"
    elif p3 == 0.0:
        verdict = "never solved - fix the agent, not the sampling"
    elif gap > 0:
        verdict = "UNRELIABLE: +%.2f from retrying (needs a runtime verifier)" % gap
    else:
        verdict = "-"
    print("%-12s pass@1 %.2f -> pass@3 %.2f   %s" % (t["id"], p1, p3, verdict))
print("-" * 58)

what retrying would buy, per task
----------------------------------------------------------
salary       pass@1 1.00 -> pass@3 1.00   already reliable - retries buy nothing
headcount    pass@1 1.00 -> pass@3 1.00   already reliable - retries buy nothing
combined     pass@1 1.00 -> pass@3 1.00   already reliable - retries buy nothing
compare      pass@1 1.00 -> pass@3 1.00   already reliable - retries buy nothing
----------------------------------------------------------


### 4. The cost of pass@k, and the honesty rule

pass@3 costs 3x pass@1. That is not free, and it is not the whole cost: to *use*
pass@3 in production you also need a runtime verifier, and building one is usually
harder than the agent.

So the rule when reporting:

- Quote **pass@1** as the operating number unless you actually retry in production.
- Quote **pass@k** alongside it to show headroom, with k, n and the task count.
- Never quote pass@k without n. `pass@5` from 5 samples is a different (and worse)
  estimate than `pass@5` from 50.

### Next

Notebook 04 asks the question this notebook still cannot answer: **how much does
this score move when nothing changes?**